All the possible options are as follows:

- Treebank
    - Nodes we're looking at (so pattern and patterns.txt file for grex)
    - Type of matrix
        - coverage
        - precision
        - geometric mean
        - pmi
        - tfidf
    - Simple data visualisation
        - PCA
        - TSNE
    - Clustering
        - Hierarchical
        - DBScan
    - Outlier detection
        - LOF
        - SOD

In [13]:
import importlib
importlib.reload(tod)
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

# UD_French-GSD - All nodes, clustered by lemma and upos

In [14]:
import plotly.express as px
import plotly.graph_objects as go

treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_Umbrian-IKUVINA"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "../3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

In [15]:
treebank_name = treebank_path.split("/")[-1]

In [ ]:
from pathlib import Path

types_of_matrix = ["precision", "coverage", "PMI", "tf-idf", "geometric_mean"]

for type_of_matrix in types_of_matrix:
    corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    matrix_type=type_of_matrix
    min_occurrences=0,
)
    dim_reds = {"pca": tod.dimension_reduction_classic.Pca_corpus(corpus), "tsne": tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)}
    clusterings = {"hierarchical": tod.clustering.HierarchicalClustering(corpus), "DBSCAN": tod.clustering.DbScan(corpus)}
    outlier_detectors = {"lof": tod.outliers.LOF(corpus), "sod": tod.outliers.SOD(corpus, ref_set=5)}

    for dim_red, _ in dim_reds.items():
        fig = tod.plotting.lexunit_scatter_plot(corpus, dim_reds[dim_red])
        directory = Path(f"/Users/madalina/Documents/M2TAL/stage/todc_visualisation/assets/treebanks/{treebank_name}/{analysed_category}/{type_of_matrix}/plain_data_visualisation")
        directory.mkdir(parents=True, exist_ok=True)  # Create the directory if it doesn't exist

        # Define the full file path
        path = directory / f"{dim_red}_plain.html"

        # Write the figure to the HTML file
        fig.write_html(str(path))
    
        for clustering_algo, _ in clusterings.items():
            directory = Path(f"/Users/madalina/Documents/M2TAL/stage/todc_visualisation/assets/treebanks/{treebank_name}/{analysed_category}/{type_of_matrix}/cluster_data_visualisation")
            directory.mkdir(parents=True, exist_ok=True)

            fig = tod.plotting.cluster_scatter_plot(corpus, dim_reds[dim_red], clusterings[clustering_algo])
            path = directory / f"{dim_red}_{clustering_algo}_scatter_plot.html"
            fig.write_html(str(path))

            fig = tod.plotting.cluster_piechart_pos(corpus, clusterings[clustering_algo])
            path = directory / f"{clustering_algo}_piechart_pos_dropdown.html"
            fig.write_html(str(path))

            fig = tod.plotting.cluster_piechart_pos(corpus, clusterings[clustering_algo], dropdown=False)
            path = directory / f"{clustering_algo}_piechart_pos_no_dropdown.html"
            fig.write_html(str(path))

        for outlier_detector, _ in outlier_detectors.items():
            directory = Path(f"/Users/madalina/Documents/M2TAL/stage/todc_visualisation/assets/treebanks/{treebank_name}/{analysed_category}/{type_of_matrix}/outlier_data_visualisation")
            directory.mkdir(parents=True, exist_ok=True)

            fig = tod.plotting.outlier_scatter_plot(corpus, dim_reds[dim_red], outlier_detectors[outlier_detector])
            path = directory / f"{dim_red}_{outlier_detector}_scatter_plot.html"
            fig.write_html(str(path))

            fig = tod.plotting.outlier_rainy_plot(corpus, outlier_detectors[outlier_detector])
            path = directory / f"{outlier_detector}_rainy_plot.html"
            fig.write_html(str(path))

            fig = tod.plotting.outlier_dotted_line_plot(corpus, outlier_detectors[outlier_detector])
            path = directory / f"{outlier_detector}_dotted_line_plot.html"
            fig.write_html(str(path))

            fig = tod.plotting.outlier_piechart_pos(corpus, outlier_detectors[outlier_detector])
            path = directory / f"{outlier_detector}_piechart_pos_dropdown.html"
            fig.write_html(str(path))

            fig = tod.plotting.outlier_piechart_pos(corpus, outlier_detectors[outlier_detector], dropdown=False)
            path = directory / f"{outlier_detector}_piechart_pos_no_dropdown.html"
            fig.write_html(str(path))



Number of matches after filtering: 746


ValueError: perplexity must be less than n_samples

In [ ]:
corpus = tod.corpus.Corpus(
    treebank_path="/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master",
    grew_pattern="pattern{X[upos<>PUNCT]}",
    patterns_text_file="../3. probability_matrix/patterns_all_nodes.txt",
    matrix_type="coverage"
)

In [10]:
from pathlib import Path

def save_figure(fig, directory, filename):
    """Helper function to save a figure to an HTML file."""
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / filename
    fig.write_html(str(path))

def process_dim_red_visualizations(corpus, dim_reds, clusterings, outlier_detectors, base_path, type_of_matrix):
    """Process and save visualizations that depend on dimension reduction."""
    for dim_red_name, dim_red in dim_reds.items():
        # Plain data visualization
        plain_dir = base_path / type_of_matrix / "plain_data_visualisation"
        save_figure(tod.plotting.lexunit_scatter_plot(corpus, dim_red), plain_dir, f"{dim_red_name}_plain.html")

        # Clustering visualizations
        cluster_dir = base_path / type_of_matrix / "cluster_data_visualisation"
        for clustering_name, clustering in clusterings.items():
            save_figure(
                tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering),
                cluster_dir,
                f"{dim_red_name}_{clustering_name}_scatter_plot.html"
            )

        # Outlier visualizations that depend on dim_red
        outlier_dir = base_path / type_of_matrix / "outlier_data_visualisation"
        for outlier_name, outlier in outlier_detectors.items():
            save_figure(
                tod.plotting.outlier_scatter_plot(corpus, dim_red, outlier),
                outlier_dir,
                f"{dim_red_name}_{outlier_name}_scatter_plot.html"
            )

def process_non_dim_red_visualizations(corpus, clusterings, outlier_detectors, base_path, type_of_matrix):
    """Process and save visualizations that do not depend on dimension reduction."""
    # Clustering visualizations
    cluster_dir = base_path / type_of_matrix / "cluster_data_visualisation"
    for clustering_name, clustering in clusterings.items():
        save_figure(
            tod.plotting.cluster_piechart_pos(corpus, clustering),
            cluster_dir,
            f"{clustering_name}_piechart_pos_dropdown.html"
        )
        save_figure(
            tod.plotting.cluster_piechart_pos(corpus, clustering, dropdown=False),
            cluster_dir,
            f"{clustering_name}_piechart_pos_no_dropdown.html"
        )

    # Outlier visualizations
    outlier_dir = base_path / type_of_matrix / "outlier_data_visualisation"
    for outlier_name, outlier in outlier_detectors.items():
        save_figure(
            tod.plotting.outlier_rainy_plot(corpus, outlier),
            outlier_dir,
            f"{outlier_name}_rainy_plot.html"
        )
        save_figure(
            tod.plotting.outlier_dotted_line_plot(corpus, outlier),
            outlier_dir,
            f"{outlier_name}_dotted_line_plot.html"
        )
        save_figure(
            tod.plotting.outlier_piechart_pos(corpus, outlier),
            outlier_dir,
            f"{outlier_name}_piechart_pos_dropdown.html"
        )
        save_figure(
            tod.plotting.outlier_piechart_pos(corpus, outlier, dropdown=False),
            outlier_dir,
            f"{outlier_name}_piechart_pos_no_dropdown.html"
        )

# Main processing loop
types_of_matrix = ["precision", "coverage", "PMI", "tf-idf", "geometric_mean"]
base_path = Path("/Users/madalina/Documents/M2TAL/stage/todc_visualisation/assets/treebanks") / treebank_name / analysed_category

for type_of_matrix in types_of_matrix:
    corpus = tod.corpus.Corpus(
        treebank_path=treebank_path,
        grew_pattern=grew_pattern,
        patterns_text_file=patterns_text_file,
        matrix_type=type_of_matrix
    )
    dim_reds = {
        "pca": tod.dimension_reduction.Pca(corpus),
        "tsne": tod.dimension_reduction.Tsne(corpus)
    }
    clusterings = {
        "hierarchical": tod.clustering.HierarchicalClustering(corpus),
        "DBSCAN": tod.clustering.DBScan(corpus, eps=0.2)
    }
    outlier_detectors = {
        "lof": tod.outliers.LOF(corpus),
        "sod": tod.outliers.SOD(corpus)
    }

    # Process visualizations
    process_dim_red_visualizations(corpus, dim_reds, clusterings, outlier_detectors, base_path, type_of_matrix)
    process_non_dim_red_visualizations(corpus, clusterings, outlier_detectors, base_path, type_of_matrix)